In [1]:
from pathlib import Path
import importlib
import sys
import os


# Loading markov model directory as module into jupyter notebook
module_dir = Path(os.getcwd()).parent.resolve()
data_dir = module_dir / "data"

if module_dir not in sys.path:
    sys.path.insert(1, str(module_dir))
else:
    print("Module path already inserted into system paths")

try:
    from model import markov_chain
    from model import probabilistic_analysis as psa
    from model import visualization
    from model import constants
    from model import utils

    # to apply changes in modules
    importlib.reload(markov_chain)
    importlib.reload(psa)
    importlib.reload(visualization)
    importlib.reload(constants)
    importlib.reload(utils)
except ModuleNotFoundError as e:
    print(f"Unable to import module: {e.msg}")

In [2]:
# Model inputs
base_sample_size = 128
short_steps = constants.SHORT_TERM_CYCLE_COUNTS
long_steps = constants.LONG_TERM_CYCLE_COUNTS
simulation_start_point = constants.START_SIMULATION_AGE_IN_WEEK

# Loading states name from excel sheet is deprecated, now on only generate within the code blocks
start_state = "Healthy"
primary_states = [
    "Healthy",
    "Bleeding",
    "Hemarthrosis",
    "Arthropathy",
    "LT_Bleeding",
    "Death",
]
secondary_states = ["Arthropathy", "Bleeding", "Hemarthrosis", "LT_Bleeding", "Death"]

# NOTE:
# Transition matrix is dynamically generated through the *_psa functions
# To change in states need to update the psa worker functions as well to support new model schema
# NOTE:
# Newly suggested model structure:               switch
#                   [Healthy]                    ------>                     [Arthropathy]
#        |              |              |                           |              |              |
# [LT Bleeding] | [Hemarthrosis] | [Bleeding]               [LT Bleeding] | [Hemarthrosis] | [Bleeding]
#    |                  |                                          |
# [DEATH]         [Arthropathy]                                 [DEATH]

chains = {"primary": (primary_states, {}), "secondary": (secondary_states, {})}


# Define switch conditions
def arthropathy_switch_condition(step: int, state: str, chain: str, **kwargs) -> bool:
    """Determine if a switch to the secondary chain should occur based on the Arthropathy state."""
    return state == "Arthropathy" and chain == "primary"

switch_conditions = {"secondary": arthropathy_switch_condition}

In [3]:
weights = [utils.cal_body_weight(w, b=simulation_start_point) for w in range(short_steps)]

# Short term simulation
on_demand_inputs, on_demand_outputs = psa.markov_chains_psa_wrapper(
    strategy="on_demand",
    n_samples=base_sample_size,
    chains=chains,
    start_chain="primary",
    start_state="Healthy",
    steps=short_steps,
    switch_conditions=switch_conditions,
)

prophylaxis_inputs, prophylaxis_outputs = psa.markov_chains_psa_wrapper(
    strategy="prophylaxis",
    n_samples=base_sample_size,
    chains=chains,
    start_chain="primary",
    start_state="Healthy",
    steps=short_steps,
    switch_conditions=switch_conditions,
)

In [ ]:
# ---- Debug cell ----
import numpy as np

debug = True

# Graphs stores at outputs/figures/transitions
if debug:
    # on_demand simulation transition matrix graph
    for i, inputs in enumerate(np.random.choice(on_demand_inputs, size=5)):
        chains = inputs["chains"]
        primary_matrix = chains["primary"][1]
        secondary_matrix = chains["secondary"][1]
        visualization.visualize_transition_matrix(
            matrix=primary_matrix,
            states=primary_states,
            title=f"on-demand-primary-{inputs['abr']}-{inputs['ajbr']}",
        )
        visualization.visualize_transition_matrix(
            matrix=secondary_matrix,
            states=secondary_states,
            title=f"on-demand-secondary-{inputs['abr']}-{inputs['ajbr']}",
        )
    # prophylaxis simulation transition matrix graph
    for i, inputs in enumerate(np.random.choice(prophylaxis_inputs, size=5)):
        chains = inputs["chains"]
        primary_matrix = chains["primary"][1]
        secondary_matrix = chains["secondary"][1]
        visualization.visualize_transition_matrix(
            matrix=primary_matrix,
            states=primary_states,
            title=f"prophylaxis-primary-{inputs['abr']}-{int(round(inputs['ajbr']))}",
        )
        visualization.visualize_transition_matrix(
            matrix=secondary_matrix,
            states=secondary_states,
            title=f"prophylaxis-secondary-{inputs['abr']}-{int(round(inputs['ajbr']))}",
        )

In [5]:
import pandas as pd

df = pd.DataFrame(
    columns=[
        "on_demand_mean",
        "on_demand_median",
        "prophylaxis_mean",
        "prophylaxis_median",
    ],
)

on_demand_arthropathy = [
    op for op in on_demand_outputs if op.sequences.count("Arthropathy") > 0
]

# On_Demand results
od_consumption = [op.annual_factor_consumption for op in on_demand_outputs]
od_costs = [op.annual_factor_costs for op in on_demand_outputs]
od_qalys = [op.qaly for op in on_demand_outputs]

# Means
df.loc["annual iu", "on_demand_mean"] = np.mean(od_consumption)
df.loc["annual iu/kg", "on_demand_mean"] = np.mean(od_consumption) / np.mean(weights)
df.loc["annual costs ($)", "on_demand_mean"] = np.mean(od_costs)
df.loc["qaly", "on_demand_mean"] = np.mean(od_qalys)

# Medians
df.loc["annual iu", "on_demand_median"] = np.median(od_consumption)
df.loc["annual iu/kg", "on_demand_median"] = np.median(od_consumption) / np.median(
    weights
)
df.loc["annual costs ($)", "on_demand_median"] = np.median(od_costs)
df.loc["qaly", "on_demand_median"] = np.median(od_qalys)


# Prophylaxis results
pro_consumption = [op.annual_factor_consumption for op in prophylaxis_outputs]
pro_costs = [op.annual_factor_costs for op in prophylaxis_outputs]
pro_qalys = [op.qaly for op in prophylaxis_outputs]

# Means
df.loc["annual iu", "prophylaxis_mean"] = np.mean(pro_consumption)
df.loc["annual iu/kg", "prophylaxis_mean"] = np.mean(pro_consumption) / np.mean(weights)
df.loc["annual costs ($)", "prophylaxis_mean"] = np.mean(pro_costs)
df.loc["qaly", "prophylaxis_mean"] = np.mean(pro_qalys)

# Medians
df.loc["annual iu", "prophylaxis_median"] = np.median(pro_consumption)
df.loc["annual iu/kg", "prophylaxis_median"] = np.median(pro_consumption) / np.median(
    weights
)
df.loc["annual costs ($)", "prophylaxis_median"] = np.median(pro_costs)
df.loc["qaly", "prophylaxis_median"] = np.median(pro_qalys)

# ICER value
icer = (np.sum(pro_costs) - np.sum(od_costs)) / (np.sum(pro_qalys) - np.sum(od_qalys))

In [6]:
# NOTE:
# QALYs values are far less than expectation, which seems there is a mistake some where on markov chain switching
# Transition matrix generator or utility reward function implementation.
# Debug before continuing
print(
    f"""
    [Short Term]
    Simulation results for {int(short_steps/52)} Years (2 - 13)
    {len(on_demand_inputs)} samples
    
    Number of simulations transitioned to chronic arthropathy: {len(on_demand_arthropathy)}
    Incremental cost effectiveness ratio: ${icer:,}
    """
)
df.head()


    [Short Term]
    Simulation results for 10 Years (2 - 13)
    2816 samples
    
    Number of simulations transitioned to chronic arthropathy: 223
    Incremental cost effectiveness ratio: $12,503.12155394044
    


,on_demand_mean,on_demand_median,prophylaxis_mean,prophylaxis_median
annual iu,14397.487642,8626.65,100387.62326,98098.1
annual iu/kg,593.190084,362.616646,4136.064858,4123.501471
annual costs ($),7126.860828,4270.254331,49692.601767,48559.271144
qaly,4.379564,3.751779,7.783973,8.959135
